# Task fMRI tutorial

This exercise generates a tiny dataset with two separately labeled runs, known event timing and synthetic motion confounds. It is a walkthrough of the public BOLD entry point. The NIfTI files and derivatives are generated by the repository, with no access to human scans.

For a public human-data exercise use the [HCP Motor and OpenNeuro guide](../docs/PUBLIC_DATA.md). To process a reviewed case, open [Task_BOLD.ipynb](Task_BOLD.ipynb).

In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys
from uuid import uuid4

repo = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "run_bold.py").is_file()), None)
if repo is None:
    raise RuntimeError("Start Jupyter from the repository root.")
tutorial_root = repo.parent / f"neo_bold_tutorial_{uuid4().hex[:8]}"

def run_bold(stage, *arguments):
    """Call the public CLI and display its failure reason if it stops."""
    completed = subprocess.run(
        [sys.executable, "-B", str(repo / "run_bold.py"), stage, *map(str, arguments)],
        cwd=repo, env={**os.environ, "PYTHONIOENCODING": "utf-8", "MPLBACKEND": "Agg"},
        capture_output=True, encoding="utf-8", check=False,
    )
    if completed.returncode:
        raise RuntimeError(completed.stdout or completed.stderr)
    return json.loads(completed.stdout)

demo = run_bold("demo", "--output", tutorial_root)
config = Path(demo["config"])
print(config)

## 1. Read the acquisition and events

Each BOLD file is one saved time series. Each run has its own events, mask and confounds. Check that the frame count and TR agree with the event timing. Distinguish acquisition labels from task conditions.

In [ ]:
inventory = run_bold("inventory", "--config", config)
display(inventory)

## 2. Register the input derivatives

The tutorial uses a marked engineering fixture. For a real case, the preprocessing must already have run and the files must be registered with actual completion and timing evidence. This stage records the input identity.

In [ ]:
registration = run_bold("preprocess", "--config", config)
display(registration)

## 3. Audit the design

Look at estimability, residual degrees of freedom and overlap between movement spikes and the design. Stop when the event clock or spatial correspondence is unresolved.

In [ ]:
audit = run_bold("audit", "--config", config)
audit_rows = json.loads(Path(audit["output"]).read_text(encoding="utf-8"))
display(audit_rows)

## 4. Fit separately, then compare saved beta estimates

The canonical HRF with a temporal derivative is the static model. DoubleScanner searches lag and duration, while FIR checks sensitivity to response shape. Compare beta estimates on a common physical grid. The two synthetic tasks exercise that path and their beta difference is descriptive.

In [ ]:
if audit["ready_runs"] != 2:
    raise RuntimeError("Review the design audit before fitting.")
display(run_bold("glm", "--config", config))
display(run_bold("compare", "--config", config))

## 5. Read the HTML report

Open the printed file in your browser. Review spatial overlays, design quality, and the model labels. A/B exploratory and stability maps answer different questions. The synthetic output confirms execution and report generation.

In [ ]:
report = run_bold("report", "--config", config)
print(Path(report["output"]).resolve())

## Optional public-data exercise

For HCP Motor, choose one participant and one LR or RL acquisition. Keep each acquisition's EVs and record the release. Compare onset, duration and condition with the scan clock as described in the [HCP intake guide](../docs/PUBLIC_DATA.md#hcp-young-adult).

For a no-account BIDS timing example, set the switch below to load the published event table for OpenNeuro `ds000117` v2.0.0. This fetches only the event table. The publisher removed two dummy volumes before time zero. Identify stimulus and button-response labels before planning a contrast. Use your verified derivatives for any real fit.

In [ ]:
load_public_events = False  # Enable only when you want to fetch the small public TSV.
if load_public_events:
    from collections import Counter
    import csv
    import hashlib
    import io
    from urllib.request import urlopen

    base = "https://raw.githubusercontent.com/OpenNeuroDatasets/ds000117/2.0.0/"
    run = "sub-01/ses-mri/func/sub-01_ses-mri_task-facerecognition_run-01"
    with urlopen(base + run + "_events.tsv", timeout=15) as source:
        payload = source.read()
    expected = "3c1d41eee81f8e362640882731be32807d1929ed7e114b9bd7963f62db4f94f9"
    if hashlib.sha256(payload).hexdigest() != expected:
        raise ValueError("The event table differs from the pinned release.")
    rows = list(csv.DictReader(io.StringIO(payload.decode("utf-8")), delimiter="\t"))
    display(Counter(row["trial_type"] for row in rows))
    last_event = max(float(row["onset"]) + float(row["duration"]) for row in rows)
    print(f"{len(rows)} events, last end at {last_event:.3f} s, {208 * 2} s of BOLD acquisition")